# Fallback: BGE-M3 embeddings on Colab

수업용 embedding API가 안 될 때만 사용합니다. Colab 안에서 BGE-M3를 직접 실행해서 issue dense index와 graph similarity edge 포함 graph를 만듭니다.

In [ ]:
!pip -q install sentence-transformers numpy


In [ ]:
from pathlib import Path

CASE_ISSUES_PATH = Path('/content/case_issues.jsonl')
ISSUE_FEATURES_PATH = Path('/content/issue_features.jsonl')

ISSUE_INDEX_OUTPUT = Path('/content/issue_embedding_index.npz')
ISSUE_METADATA_OUTPUT = Path('/content/issue_embedding_metadata.json')
GRAPH_OUTPUT = Path('/content/issue_graph_with_similarity_fallback.json')

MODEL_NAME = 'BAAI/bge-m3'
SIMILARITY_THRESHOLD = 0.75
SIMILARITY_TOP_K = 5


In [ ]:
import json, re
from collections import Counter
import numpy as np
from sentence_transformers import SentenceTransformer

def load_jsonl(path):
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_phrase(value):
    value = re.sub(r'\s+', ' ', str(value).strip())
    return value.strip('.,;:()[]{}\"\'')

def iter_issue_records(rows):
    for row in rows:
        document_id = row.get('document_id', '')
        for issue_index, issue in enumerate(row.get('issues', []) or []):
            yield {
                'document_id': document_id,
                'issue_index': issue_index,
                'case_title': row.get('case_title', ''),
                'decision_type': row.get('decision_type', ''),
                'tax_item': row.get('tax_item', ''),
                'issue': issue.get('issue', ''),
                'taxpayer_argument': issue.get('taxpayer_argument', ''),
                'judgment_reasoning': issue.get('judgment_reasoning', ''),
                'conclusion': issue.get('conclusion', ''),
            }

def issue_embed_text(record):
    parts = []
    for key, label in [('tax_item', '세목'), ('case_title', '판시사항'), ('issue', '쟁점'), ('taxpayer_argument', '납세자 주장')]:
        if record.get(key):
            parts.append(f'[{label}] {record[key]}')
    return '\n'.join(parts)

model = SentenceTransformer(MODEL_NAME)


## 1. Issue dense index

`case_issues.jsonl`을 업로드한 뒤 실행합니다. 출력 파일 두 개를 repo의 `data/indexes/` 아래로 옮기면 `scripts/08_dense_issue_search.py`가 그대로 읽을 수 있습니다.

In [ ]:
records = list(iter_issue_records(load_jsonl(CASE_ISSUES_PATH)))
texts = [issue_embed_text(record) for record in records]
embeddings = model.encode(texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True).astype('float32')
np.savez_compressed(ISSUE_INDEX_OUTPUT, embeddings=embeddings)

metadata = [{**record, 'embedding_text': text, 'embedding_model': MODEL_NAME} for record, text in zip(records, texts)]
ISSUE_METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('issues=', len(records), 'dim=', embeddings.shape[1])
print(ISSUE_INDEX_OUTPUT)
print(ISSUE_METADATA_OUTPUT)


## 2. Graph with SIMILAR_TO edges

`issue_features.jsonl`을 업로드한 뒤 실행합니다. LegalConcept, FactPattern, EvidenceType 안에서만 phrase similarity edge를 만듭니다.

In [ ]:
def node_id(node_type, label):
    return f'{node_type.lower()}::{label}'

def add_node(nodes, node_type, label, **attrs):
    nid = node_id(node_type, label)
    nodes[nid] = {'id': nid, 'type': node_type, 'label': label, **attrs}
    return nid

rows = load_jsonl(ISSUE_FEATURES_PATH)
nodes, edges = {}, []
phrase_by_type = {'LegalConcept': set(), 'FactPattern': set(), 'EvidenceType': set()}

for row in rows:
    case = add_node(nodes, 'Case', row['document_id'])
    issue = add_node(nodes, 'Issue', f"{row['document_id']}::{row['issue_index']}", text=row.get('issue', ''))
    edges.append({'src': case, 'dst': issue, 'relation': 'HAS_ISSUE'})
    outcome_value = normalize_phrase(row.get('outcome') or row.get('conclusion') or '')
    if outcome_value:
        outcome = add_node(nodes, 'Outcome', outcome_value)
        edges.append({'src': issue, 'dst': outcome, 'relation': 'HAS_OUTCOME'})
    for field, node_type, relation in [
        ('legal_concepts', 'LegalConcept', 'INVOLVES_CONCEPT'),
        ('fact_patterns', 'FactPattern', 'HAS_FACT_PATTERN'),
        ('evidence_types', 'EvidenceType', 'HAS_EVIDENCE_TYPE'),
    ]:
        for value in row.get(field, []) or []:
            label = normalize_phrase(value)
            if not label:
                continue
            target = add_node(nodes, node_type, label)
            phrase_by_type[node_type].add(label)
            edges.append({'src': issue, 'dst': target, 'relation': relation})

for node_type, phrases in phrase_by_type.items():
    phrases = sorted(phrases)
    if len(phrases) < 2:
        continue
    emb = model.encode(phrases, batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    sim = emb @ emb.T
    for i, src_phrase in enumerate(phrases):
        scores = sim[i].copy()
        scores[i] = -1
        for j in np.argsort(scores)[::-1][:SIMILARITY_TOP_K]:
            if scores[j] >= SIMILARITY_THRESHOLD:
                edges.append({
                    'src': node_id(node_type, src_phrase),
                    'dst': node_id(node_type, phrases[j]),
                    'relation': 'SIMILAR_TO',
                    'weight': float(scores[j]),
                })

graph = {'nodes': list(nodes.values()), 'edges': edges}
GRAPH_OUTPUT.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding='utf-8')
print('nodes=', len(graph['nodes']), 'edges=', len(graph['edges']))
print('relations=', Counter(edge['relation'] for edge in graph['edges']))
print(GRAPH_OUTPUT)
